# Keystone-Σ0 PLT — Stage 0 Parity (Colab, ≥24 GB GPU)

The **blocking gate** before any training (ADR-0011): prove our hand-ported PLT
forward (`modeling_keystone_plt.py`) loads the Apache-2.0 LoopCoder-V2 weights and
generates sane code. This is the *definitive* bf16 run that complements the local
RTX-3070 4-bit smoke.

**Pick a GPU runtime first:** Runtime → Change runtime type → A100 / L4 (≥24 GB → bf16).
A free T4 (16 GB) still works but falls back to 4-bit.

Design: `models/keystone-sigma0-plt/ADAPTIVE-LOOP-GATE.md` · Runbook: `README.md`

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
# Deps. torch is preinstalled on Colab with the right CUDA — do NOT reinstall it.
!pip install -q "transformers==4.57.1" "huggingface_hub>=0.25" "accelerate>=1.0" \
    "bitsandbytes>=0.44" "peft>=0.13" "safetensors>=0.4" "sentencepiece>=0.2"

In [ ]:
# Get the PLT package. If the repo is PRIVATE, paste a GitHub token (repo scope);
# if public, just press Enter. LFS blobs are skipped (we only need the .py files).
import os, getpass, subprocess
TOKEN = getpass.getpass('GitHub token (blank if public): ').strip()
PUB = 'https://github.com/alex-place/lantern-os.git'
url = f'https://{TOKEN}@github.com/alex-place/lantern-os.git' if TOKEN else PUB
os.environ['GIT_LFS_SKIP_SMUDGE'] = '1'
if not os.path.isdir('lantern-os'):
    subprocess.run(['git', 'clone', '--depth', '1', url, 'lantern-os'], check=True)
    subprocess.run(['git', '-C', 'lantern-os', 'remote', 'set-url', 'origin', PUB])  # scrub token
%cd lantern-os/models/keystone-sigma0-plt

In [ ]:
# Build the loadable checkpoint: download ~18 GB Apache-2.0 weights, drop our code
# in, patch config.json auto_map -> our modeling class. One time, ~10-20 min.
!python download_and_patch.py --out ./checkpoint

In [ ]:
# STAGE-0 GATE. bf16 on >=24 GB (truest smoke), else 4-bit. PASS requires:
#   loaded + 0 missing/unexpected weight keys + >=2/3 coherent generations.
import torch
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
dtype = 'bf16' if vram >= 24 else '4bit'
print(f'GPU ~{vram:.0f} GB  ->  check_parity --dtype {dtype}')
!python check_parity.py --model ./checkpoint --dtype {dtype} --max-new-tokens 96

In [ ]:
# Verdict log (one JSON line per run).
!tail -n 3 ../../data/convergence/keystone-plt-parity-log.jsonl

## Definitive parity (optional, the real proof)

The smoke above proves *loads + generates*. The **faithful** check is logit parity
vs the upstream vLLM fork (`yxing-bj/vllm`, ≥24 GB):

1. In a vLLM-fork env, run LoopCoder-V2 on a fixed `input_ids`, save
   `{"input_ids", "logits"}` (last-step, `[1,T,V]`) to `ref_logits.pt`.
2. `!python check_parity.py --model ./checkpoint --dtype bf16 --ref ref_logits.pt`
3. **PASS = `top1_agree >= 0.99`.** If it fails, the cause is one of the three
   reconstructed boundaries (CLP shift / sliding-window edge / per-loop norm) — all
   isolated + commented in `modeling_keystone_plt.py`.

**On PASS:** proceed to Stage 1 — the Adaptive Loop Gate (`ADAPTIVE-LOOP-GATE.md`),
depth ∈ [0,2], adapter-only on a frozen base. Do **not** train until this is PASS.